# Notebook 03 — Trajectory Visualizer
Shows exactly WHERE on each shape the user deviates from the target path.

| Audience | What they get |
|---|---|
| **User / Patient** | See your own drawing vs the target — visual self-feedback |
| **Doctor / Clinician** | Spatial deviation map = where motor control breaks down |
| **App maker** | Which checkpoints have highest miss rate → tune scoring thresholds |


In [1]:
DATA_DIR     = '.'
OUT_DIR      = 'outputs'
GROUP_LABEL  = 'Group A'
CANVAS       = 500
TARGET_SHAPE = None   # set to e.g. 'CIRCLE' to plot one shape; None = all
N_INTERP     = 100
import os, json, sys, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
warnings.filterwarnings('ignore')

# ── Make sure output folder exists BEFORE anything tries to write to it ──
os.makedirs(OUT_DIR, exist_ok=True)

sys.path.insert(0, os.path.dirname(os.path.abspath('hci_utils.py')))
from hci_utils import (load_board_tries, load_bd_sessions, load_piano_sessions,
                        load_piano_movements, compare_groups, save_fig,
                        SHAPE_ORDER, HAND_COLORS, GROUP_COLORS)

from scipy.interpolate import interp1d
df = load_board_tries(DATA_DIR)
shapes = ([TARGET_SHAPE] if TARGET_SHAPE
          else [s for s in SHAPE_ORDER if s in df['shapeType'].values])
print(f'Shapes to plot: {shapes}')


KeyError: 'startedAt'

## Helper functions

In [ ]:
def path_to_array(coords):
    if not coords:
        return np.empty((0, 2))
    return np.array([[p['x'], p['y']] for p in coords])

def interpolate_path(pts, n=100):
    if len(pts) < 2:
        return pts
    dists = np.cumsum(np.r_[0, np.hypot(np.diff(pts[:,0]), np.diff(pts[:,1]))])
    if dists[-1] == 0:
        return pts
    t_new = np.linspace(0, dists[-1], n)
    fx = interp1d(dists, pts[:,0], kind='linear')
    fy = interp1d(dists, pts[:,1], kind='linear')
    return np.column_stack([fx(t_new), fy(t_new)])


## Fig 03a — Trajectory overlay per shape

In [ ]:
from matplotlib.lines import Line2D

for shape in shapes:
    sub = df[df['shapeType'] == shape]
    if sub.empty:
        print(f'No data for {shape}')
        continue

    fig, ax = plt.subplots(figsize=(5, 5))
    bg = path_to_array(sub.iloc[0]['bgCoordinates'])
    if len(bg):
        ax.plot(bg[:,0]*CANVAS, bg[:,1]*CANVAS, color='black', lw=3, label='Target')

    completed_count = 0
    failed_count = 0
    for _, row in sub.iterrows():
        gc = path_to_array(row.get('gameCoordinates', []))
        if len(gc) < 2:
            continue
        color = '#2a6e4f' if row['completed'] else '#c84b2f'
        ax.plot(gc[:,0]*CANVAS, gc[:,1]*CANVAS, color=color, lw=1.2, alpha=0.45)
        if row['completed']:
            completed_count += 1
        else:
            failed_count += 1

    legend_els = [
        Line2D([0],[0], color='black', lw=3, label='Target'),
        Line2D([0],[0], color='#2a6e4f', lw=2, alpha=0.7, label=f'Completed ({completed_count})'),
        Line2D([0],[0], color='#c84b2f', lw=2, alpha=0.7, label=f'Not done ({failed_count})'),
    ]
    ax.legend(handles=legend_els, fontsize=8)
    ax.set_xlim(0, CANVAS)
    ax.set_ylim(CANVAS, 0)
    ax.set_aspect('equal')
    ax.axis('off')
    ax.set_title(GROUP_LABEL + ' — Trajectory overlay: ' + shape, fontweight='bold')
    plt.tight_layout()
    save_fig(fig, 'fig03a_trajectory_overlay_' + shape.lower() + '.png', OUT_DIR)
    plt.show()
    print(f'{shape}: {completed_count} completed, {failed_count} not completed')


## Fig 03b — Checkpoint deviation heatmap per shape
Red = users deviate most here. Green = users stay close to target.

In [ ]:
import matplotlib.cm as cm

deviation_report = []
for shape in shapes:
    sub = df[df['shapeType'] == shape]
    if sub.empty:
        continue
    bg = path_to_array(sub.iloc[0]['bgCoordinates'])
    if len(bg) < 2:
        continue
    bg_interp = interpolate_path(bg, N_INTERP)
    deviations = []
    for _, row in sub.iterrows():
        gc = path_to_array(row.get('gameCoordinates', []))
        if len(gc) < 2:
            continue
        gc_interp = interpolate_path(gc, N_INTERP)
        dist = np.hypot(gc_interp[:,0] - bg_interp[:,0],
                        gc_interp[:,1] - bg_interp[:,1])
        deviations.append(dist)
    if not deviations:
        print(f'No gameCoordinates for {shape}')
        continue

    mean_dev  = np.mean(deviations, axis=0)
    worst_idx = int(np.argmax(mean_dev))
    deviation_report.append({
        'shape': shape,
        'max_deviation_checkpoint': worst_idx,
        'max_deviation_value': float(mean_dev[worst_idx]),
        'mean_deviation': float(mean_dev.mean()),
    })

    norm_dev = mean_dev / (mean_dev.max() + 1e-9)
    fig, ax = plt.subplots(figsize=(5, 5))
    cmap_fn  = cm.get_cmap('RdYlGn_r')
    for i in range(len(bg_interp) - 1):
        ax.plot(
            [bg_interp[i,0]*CANVAS, bg_interp[i+1,0]*CANVAS],
            [bg_interp[i,1]*CANVAS, bg_interp[i+1,1]*CANVAS],
            color=cmap_fn(norm_dev[i]), lw=5, solid_capstyle='round'
        )
    sm = plt.cm.ScalarMappable(
        cmap='RdYlGn_r',
        norm=plt.Normalize(mean_dev.min(), mean_dev.max()))
    sm.set_array([])
    plt.colorbar(sm, ax=ax, fraction=0.03, pad=0.02, label='Mean deviation')
    ax.set_xlim(0, CANVAS)
    ax.set_ylim(CANVAS, 0)
    ax.set_aspect('equal')
    ax.axis('off')
    ax.set_title(GROUP_LABEL + ' — Deviation heatmap: ' + shape
                 + '\n(green=accurate, red=problematic)', fontweight='bold')
    plt.tight_layout()
    save_fig(fig, 'fig03b_checkpoint_deviation_' + shape.lower() + '.png', OUT_DIR)
    plt.show()

if deviation_report:
    dev_df = pd.DataFrame(deviation_report)
    dev_df.to_csv(os.path.join(OUT_DIR, 'trajectory_deviation_report.csv'), index=False)
    print('\n=== CLINICIAN SIGNAL ===')
    print('Checkpoint with highest deviation = specific part of the movement that breaks down.')
    print('Compare this across sessions to see if that specific weakness is resolving.')
    print(dev_df.to_string(index=False))


## Fig 03c — Drawing speed along path

In [ ]:
for shape in shapes:
    sub = df[df['shapeType'] == shape]
    has_ts = sub.apply(
        lambda r: any('timestamp' in p for p in (r.get('gameCoordinates') or [])),
        axis=1
    )
    sub_ts = sub[has_ts]
    if sub_ts.empty:
        print(f'No timestamp data for {shape} — '
              'speed analysis needs timestamp field in gameCoordinates')
        continue

    fig, ax = plt.subplots(figsize=(8, 3))
    for _, row in sub_ts.iterrows():
        gc  = row['gameCoordinates']
        pts = np.array([[p['x'], p['y'], p['timestamp']]
                        for p in gc if 'timestamp' in p])
        if len(pts) < 2:
            continue
        dx = np.diff(pts[:,0])
        dy = np.diff(pts[:,1])
        dt = np.diff(pts[:,2])
        dt[dt == 0] = 1e-9
        speed = np.hypot(dx, dy) / dt
        t_mid = (pts[:-1,2] + pts[1:,2]) / 2
        ax.plot(t_mid, speed, alpha=0.5, lw=1.5)
    ax.set_xlabel('Timestamp')
    ax.set_ylabel('Speed (canvas units / time)')
    ax.set_title(GROUP_LABEL + ' — Drawing speed: ' + shape, fontweight='bold')
    plt.tight_layout()
    save_fig(fig, 'fig03c_drawing_speed_' + shape.lower() + '.png', OUT_DIR)
    plt.show()
    print('=== CLINICIAN SIGNAL ===')
    print('Sharp speed drops at specific path positions = motor hesitation / tremor.')
